<a href="https://colab.research.google.com/github/2303A51553/HPC/blob/main/2303A51553_16_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%%bash

cat <<EOF > mpi_sum.c
#include <stdlib.h>
#include <mpi.h>
#include<stdio.h>
#define START 0
#define END 100

int main(int argc, char** argv){
    int rank, size;

    /*initialize MPI */
    MPI_Init(&argc, &argv);

    /*find out process rank (process id) and size (no. of processes)*/
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    /* calculate the start and end points by evenly dividing the range */
    int start = ((END-START)/size)*rank;
    int end =start+((END-START)/size)-1;

    /*The last process needs to do all remaining ones*/
    if(rank==size-1){
        end=END;
    }
    /* do te calculation*/
    int sum=0,i;
    for(i=start;i<=end;i++){
        sum+=i;
    }

    /*degugging output*/
    printf("process %d:sum(%d, %d)=%d\n",rank,start,end,sum);

    if(rank==0){
        int partial,i;
        MPI_Status status;
        for(i=1;i<size;i++){
            MPI_Recv(&partial,1,MPI_INT,MPI_ANY_SOURCE,0,MPI_COMM_WORLD,&status);
            printf("process 0 got data from process %d.\n",status.MPI_SOURCE);
            sum+=partial;
        }
        }else {
            MPI_Send(&sum,1,MPI_INT,0,0,MPI_COMM_WORLD);
        }
    if(rank==0){
        printf("The final sum=%d. \n",sum);


    }
    MPI_Finalize();
    return 0;
}
EOF

# Compile the C code using mpicc (MPI C compiler)
mpicc mpi_sum.c -o mpi_sum

# Check if compilation was successful
if [ $? -eq 0 ]; then
    echo "Compilation successful. Running with 4 processes:"
    # Run the MPI program with 4 processes (you can change this number)
    mpirun --allow-run-as-root --oversubscribe -np 4 ./mpi_sum
else
    echo "Compilation failed."
fi


Compilation successful. Running with 4 processes:
process 0:sum(0, 24)=300
process 3:sum(75, 100)=2275
process 2:sum(50, 74)=1550
process 1:sum(25, 49)=925
process 0 got data from process 3.
process 0 got data from process 1.
process 0 got data from process 2.
The final sum=5050. 


In [15]:
%%writefile sum-reduction.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char** argv) {
    int rank, size;
    int local_sum = 0;
    int global_sum = 0;
    int i;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // Each process calculates a local sum
    // For simplicity, let's say each process sums its rank
    local_sum = rank;
    printf("Process %d: local_sum = %d\n", rank, local_sum);

    // Use MPI_Reduce to sum all local_sum values into global_sum at root process (rank 0)
    MPI_Reduce(&local_sum, &global_sum, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("Global sum on process 0 = %d\n", global_sum);
    }

    MPI_Finalize();
    return 0;
}

Overwriting sum-reduction.c


In [16]:
!mpicc sum-reduction.c -o sum-reduction

In [17]:
!mpiexec --allow-run-as-root --oversubscribe -np 4 ./sum-reduction

Process 2: local_sum = 2
Process 1: local_sum = 1
Process 0: local_sum = 0
Process 3: local_sum = 3
Global sum on process 0 = 6


In [23]:

%%writefile broadcast.c

cat <<EOF > mpi_bcast.c
#include <stdio.h>
#include <mpi.h>
int main(int argc, char** argv){
    int rank, size;

    /*initialize MPI */
    MPI_Init(&argc, &argv);

    /*find out process rank (process id) and size (no. of processes)*/
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    char string[100];
    if(rank == 0){
        printf("Enter a string: ");
        fflush(stdout);
        scanf("%s", string);
    }
    MPI_Bcast(string, 100, MPI_CHAR, 0, MPI_COMM_WORLD);
    printf("Process %d has %s!\n",rank,string);
    MPI_Finalize();
    return 0;
}
EOF

# Compile the C code using mpicc (MPI C compiler)
mpicc mpi_bcast.c -o mpi_bcast

# Check if compilation was successful
if [ $? -eq 0 ]; then
    echo "Compilation successful. Running with 4 processes:"
    # Run the MPI program with 4 processes
    mpirun --allow-run-as-root --oversubscribe -np 4 ./mpi_bcast
else
    echo "Compilation failed."
fi


Writing broadcast.c


In [24]:
!mpicc broadcast.c -o broadcast

broadcast.c:2:5: error: expected ‘=’, ‘,’, ‘;’, ‘asm’ or ‘__attribute__’ before ‘<<’ token
    2 | cat <<EOF > mpi_bcast.c
      |     ^~
In file included from /usr/include/stdio.h:43,
                 from broadcast.c:3:
/usr/include/x86_64-linux-gnu/bits/types/struct_FILE.h:95:3: error: unknown type name ‘size_t’
   95 |   size_t __pad5;
      |   ^~~~~~
/usr/include/x86_64-linux-gnu/bits/types/struct_FILE.h:98:67: error: ‘size_t’ undeclared here (not in a function)
   98 |   char _unused2[15 * sizeof (int) - 4 * sizeof (void *) - sizeof (size_t)];
      |                                                                   ^~~~~~
/usr/include/x86_64-linux-gnu/bits/types/struct_FILE.h:1:1: note: ‘size_t’ is defined in header ‘<stddef.h>’; did you forget to ‘#include <stddef.h>’?
  +++ |+#include <stddef.h>
    1 | /* Copyright (C) 1991-2022 Free Software Foundation, Inc.
In file included from broadcast.c:3:
/usr/include/stdio.h:308:35: error: expected declaration specifiers or ‘...’ bef

In [ ]:
!mpiexe